# Basic LLM Agents

Today you turn the retrieval you built this morning into a tool, inside an agent
you write yourself.

```text
question -> LLM -> tool call -> execute -> observation -> LLM -> ... -> answer
```

**Rules**

1. No frameworks: LangChain, LangGraph, LlamaIndex agents, CrewAI, AutoGen,
   OpenAI Agents SDK. The provider's native tool calling is fine. The loop is
   written by hand.
2. Opik is on from the first cell. It is not a separate chapter. This is how the
   work is done.
3. When something fails, read the trace before you touch the prompt. Look for
   the **first** wrong step, not the last wrong sentence.
4. No explicit planning. That is tomorrow.

Cells are marked **PROVIDED** (read it, run it, move on), **TODO** (you write
it), or 🔴 **PLACEHOLDER** (must be filled in before the exercise is handed out).

---

## 0. Fill in before teaching

This notebook is a scaffold. Five things are deliberately empty, because they
depend on the story, on the provider, and on what you hand the students.
Everything else (the agent loop, dispatch, evaluation, failure analysis) is
domain-independent.

| # | Placeholder | Where | What it needs |
|---|---|---|---|
| 1 | Scenario | §1 | The operational scenario, continuing this morning's exercise |
| 2 | Corpus | §3.1 | 10–15 documents |
| 3 | Provider adapter | §2.2 | Three functions that talk to the provider's API |
| 4 | Capabilities | §4.1 | The API / client / functions students receive and wrap |
| 5 | Dev set | §10.1 | 10–15 questions across seven categories |

Search the notebook for `PLACEHOLDER`.

**Content constraint.** What you provide in §4 must hold facts that appear in no
document, and vice versa. If one source answers everything, the agent never has
to *choose*, and the exercise collapses into RAG with extra steps.

**Dependency chain.** A document mentions an entity by id → the capability
resolves it → a further call completes the answer. That chain is what makes §9
real, and it is exactly what you cannot hard-code in advance.

---

## 1. Mission

> 🔴 **PLACEHOLDER: write the scenario here**
>
> A direct continuation of this morning: same organisation, same archive.
>
> **The narrative beat:** this morning's retrieval worked. Then people started
> asking questions whose answer simply is not in the documents, but in another
> source. Answering takes two sources, in an order that depends on what the
> first call returned. You cannot fix the order in advance, because the next
> question will need a different one.
>
> **The task:** give the model tools, and let it decide which source to use, in
> which order, and when it has enough to answer.

### 🔴 Example questions

| # | Question | What it needs |
|---|------|---------|
| 1 | *«replace: answered from the documents alone»* | documents |
| 2 | *«replace: answered from one capability call»* | capability |
| 3 | *«replace: a document, then a capability keyed on what it returned»* | documents → capability |
| 4 | *«replace: two different capability calls»* | two calls |
| 5 | *«replace: general knowledge or a writing task»* | no tool |

Row 5 stays. An agent that calls a tool on every question is a slow, expensive
agent. Deciding **not** to act is part of the job.

---

## Before you start

Fifteen minutes, before you write any code. Nothing to run, nothing graded.
These are theory questions about agents in general — none of them need anything
from this notebook. Several come back in §12 and §13; the point is to commit to
an answer now, while being wrong is still free.

**What an agent is**

1. On one particular question, a fixed pipeline and an agent make the exact same
   sequence of calls, in the same order. Something is still different between
   them. What? Your answer should not be about the calls.
2. Define "agent" in one sentence, without using the words *autonomous*,
   *reasoning*, or *intelligent*.

**The loop**

```text
while not done:
    action      = LLM(state)
    observation = environment(action)
    state      += observation
```

3. Three of *state*, *action*, *observation*, *done* are data. One is a
   decision. Which one, and who makes it?
4. What if the model never stops asking for tools? Name two ways a system can
   end that, and what each one costs you.
5. Should an agent be able to decide to do nothing at all? What does it cost if
   it cannot?

**Tools**

6. The model never sees a tool's source code. What *does* it see, and what does
   it have to infer?
7. Two tools are called `search` and `lookup`, both described as "find
   information". On what basis does the model choose between them? What is the
   smallest change that would reliably flip the choice?
8. A tool raises an exception. Three options: let it kill the run, hand the
   error back to the model as text, or retry silently. Pick one, and say what
   the other two would cost you.

**Memory**

9. Between two steps an agent is nothing but a stateless API call. So where does
   what-it-has-already-done physically live?
10. An agent takes 40 steps. It makes its mistake at step 6, and the mistake
    only becomes visible at step 30. What about this design makes that hard to
    locate, and what would you want recorded to make it easy?

**Judging one**

11. An agent reaches the correct answer by a route that makes no sense. Does it
    pass?
12. Agent A answers 90% of questions correctly and uses 8 tool calls per
    question. Agent B answers 85% and uses 2. Which would you ship, and what
    else would you need to know to be sure?

**Predictions — write these down**

You will check them later. A wrong prediction you recorded is worth more than a
right one you did not.

13. You are about to give a model two sources: a document archive and a
    structured lookup. Which do you predict it will over-use, and why?
14. A tool finds nothing and returns an empty string. What do you expect the
    model to do next?
15. You may not change the model today. List three things you *can* change that
    you expect to move an agent's behaviour, ranked by how much.

---

## 2. Setup

§2.1 and §2.3 are provided. §2.2 is a placeholder.

In [ ]:
# PROVIDED - dependencies. Run once if you need to.
# %pip install --quiet opik
# %pip install --quiet <your-model-provider-sdk>
# %pip install --quiet <any client library your section-4 capabilities need>

In [ ]:
# PROVIDED - imports and Opik.
import json
import math
import os
import re
import time
from collections import Counter
from dataclasses import dataclass, field
from typing import Any, Optional

os.environ.setdefault("OPIK_PROJECT_NAME", "basic-agents-exercise")

# Two names the whole notebook uses:
#   @track(name=...)   opens a span
#   update_trace(...)  attaches a name / tags / metadata to the current trace
# If Opik is unavailable they become no-ops and the notebook still runs - but
# the entire exercise is built on reading traces. Fix it before continuing.
OPIK_ENABLED = False
try:
    import opik
    from opik import opik_context

    opik.configure(use_local=os.environ.get("OPIK_USE_LOCAL", "false").lower() == "true")
    OPIK_ENABLED = True
except Exception as exc:  # noqa: BLE001
    print("!! Opik is not active (" + type(exc).__name__ + ": " + str(exc) + ")")

if OPIK_ENABLED:
    track = opik.track

    def update_trace(**kwargs):
        try:
            opik_context.update_current_trace(**kwargs)
        except Exception:  # noqa: BLE001
            pass
else:
    def track(*args, **kwargs):
        if args and callable(args[0]):
            return args[0]
        return lambda fn: fn

    def update_trace(**kwargs):
        return None

print("Opik enabled:", OPIK_ENABLED)

### 2.1 The provider contract: PROVIDED

The loop does not need to know who the provider is. Three functions know;
everything else works against a normalised response.

```text
         provider SDK
              |
   +----------+-----------+
   |  3 adapter functions |   <- the only provider-specific code (2.2)
   +----------+-----------+
              |
   LLMResponse / ToolCall     <- provider-neutral from here down
              |
   tools, dispatch, agent loop, evaluation
```

The dataclasses below are the interface you implement against.

In [ ]:
# PROVIDED - the normalised response contract. Do not change it.
@dataclass
class ToolCall:
    """One tool call the model asked for."""
    id: str            # the provider's id - needed to return the right result
    name: str          # the tool name
    arguments: dict    # already decoded from JSON


@dataclass
class LLMResponse:
    """One model turn, normalised across providers."""
    text: str                    # visible text ("" if the model only called tools)
    tool_calls: list             # list[ToolCall]; empty = the model is done
    stop_reason: str             # the provider's stop reason, for debugging
    assistant_message: dict      # the assistant turn in provider format, to append as-is
    usage: dict = field(default_factory=dict)
    raw: Any = None              # the original object


@dataclass
class ToolResult:
    """An executed tool call, on its way back to the model."""
    tool_call_id: str
    name: str
    content: str
    is_error: bool = False

### 2.2 🔴 PLACEHOLDER: provider adapter

Implement the three functions against the provider's **native tool calling**.
Not a text protocol you parse yourself.

**`assistant_message` is the one that matters.** It is the assistant turn in the
provider's own format, returned **as-is**. Do not rebuild it from `text`. That
is where the tool-call ids and the thinking blocks live, and the next request
needs them. On some models this is an immediate API error; on others it is
silent degradation. The most common bug in this exercise.

```python
# --- OpenAI-style ------------------------------------------------------
msg = resp.choices[0].message
LLMResponse(
    text=msg.content or "",
    tool_calls=[ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments or "{}"))
                for tc in (msg.tool_calls or [])],
    stop_reason=resp.choices[0].finish_reason,
    assistant_message=msg.model_dump(exclude_none=True),
)
# one message per result:
#   {"role": "tool", "tool_call_id": r.tool_call_id, "content": r.content}

# --- Anthropic-style ---------------------------------------------------
LLMResponse(
    text="".join(b.text for b in resp.content if b.type == "text"),
    tool_calls=[ToolCall(id=b.id, name=b.name, arguments=b.input)
                for b in resp.content if b.type == "tool_use"],
    stop_reason=resp.stop_reason,
    assistant_message={"role": "assistant", "content": resp.content},
)
# one message for all results:
#   {"role": "user", "content": [{"type": "tool_result",
#                                 "tool_use_id": r.tool_call_id,
#                                 "content": r.content, "is_error": r.is_error}, ...]}
```

Note the difference in that last step. Splitting results across several messages
for a provider that wants one is an API error; for others it teaches the model
to stop issuing parallel tool calls. That is why
`make_tool_result_messages` returns a **list**.

In [ ]:
# ==========================================================================
# 🔴 PLACEHOLDER - PROVIDER ADAPTER. Implement the three functions.
# ==========================================================================

MODEL = "<<REPLACE: model id>>"
LLM_SETTINGS = {
    # 🔴 Provider parameters, in one place, so they can be tuned in §11 without
    # touching the adapter. For example "max_tokens": 8000, "temperature": 0, ...
}

# 🔴 Build the client. If the provider has an Opik integration, wrap it here and
# every API call becomes a span automatically. Otherwise the @track on call_llm
# still gives you one span per call.
client = None  # <<REPLACE>>


def to_provider_tools(tools: list) -> Any:
    """🔴 Convert the neutral tool specs from §5 into the provider's format.

    Input: [{"name": str, "description": str, "parameters": <JSON Schema>}]

    OpenAI-style:    [{"type": "function", "function": {...}}]
    Anthropic-style: [{"name":..., "description":..., "input_schema": <parameters>}]
    """
    raise NotImplementedError("to_provider_tools")


@track(name="llm_call")
def call_llm(messages: list, tools: Optional[list] = None,
             system: Optional[str] = None, **overrides) -> LLMResponse:
    """🔴 One call to the model. Returns a normalised LLMResponse.

    messages  : the conversation so far, in provider format
    tools     : neutral tool specs (pass them through to_provider_tools), or None
    system    : the system prompt, or None
    overrides : per-call overrides layered on top of LLM_SETTINGS

    Requirements: native tool calling, every LLMResponse field populated, and
    assistant_message is the assistant turn in provider format, as-is.
    """
    raise NotImplementedError("call_llm")


def make_tool_result_messages(results: list) -> list:
    """🔴 Turn list[ToolResult] into the messages to append to `messages`.

    Returns a list, so that a provider wanting one message per result and a
    provider wanting one message for all of them both fit.
    """
    raise NotImplementedError("make_tool_result_messages")


def make_user_message(text: str) -> dict:
    """🔴 Usually just {"role": "user", "content": text}."""
    return {"role": "user", "content": text}

In [ ]:
# PROVIDED - display. Depends only on LLMResponse, not on the provider.
def show(response: LLMResponse) -> None:
    print("stop_reason:", response.stop_reason)
    if response.text.strip():
        print("  [text]", response.text.strip()[:600])
    for call in response.tool_calls:
        print("  [tool_call]", call.name, json.dumps(call.arguments, ensure_ascii=False))
    if response.usage:
        print("  tokens:", response.usage)


def print_trajectory(trace: dict) -> None:
    print("Q: " + trace["question"])
    print("stop_reason=" + str(trace["stop_reason"]) + "  steps=" + str(trace["steps"]))
    for call in trace["tool_calls"]:
        flag = " [ERROR]" if call["is_error"] else ""
        print("  step " + str(call["step"]) + ": " + call["name"]
              + "(" + json.dumps(call["arguments"], ensure_ascii=False) + ")" + flag)
        print("       -> " + str(call["observation"])[:200].replace("\n", " | "))
    print("ANSWER: " + str(trace["answer"])[:800])

### 2.3 First call, first trace

Once the adapter is implemented, run this. Then **open the trace in Opik**. You
want to know where things land before there is anything interesting in there.

In [ ]:
# PROVIDED - a plain call with no tools. Go find it in Opik.
@track(name="warmup")
def warmup():
    update_trace(name="warmup", tags=["setup"])
    resp = call_llm(messages=[make_user_message("Reply with exactly: tracing works.")],
                    system="You are a concise assistant.")
    return resp.text


print(warmup())

> **Exercise 2.a:** in Opik, open the trace named `warmup`. Where does the
> system prompt appear, and where does the output? Five minutes now saves an
> hour later.

---

## 3. RAG as a tool

A tool is a Python function the model is allowed to ask you to run. That's all
it is. What makes it interesting is that **everything around the tool is part of
the prompt**:

- **name**: `search_documents` vs `search` vs `rag`
- **description**: when to reach for it, and when not to
- **arguments**: only `query`? `top_k` too?
- **what it returns**: how many passages, in what format, with what metadata

If those are wrong, no amount of system-prompt tuning will save you. This is the
first place to look when the agent picks the wrong source.

### 3.1 🔴 PLACEHOLDER: corpus

The cell below defines the **structure**, with three fake documents, so the
notebook runs. Replace the content.

- 10–15 documents is enough. A mix of types (incidents, procedures, summaries)
  gives the retriever something to tell apart.
- Keep the keys `doc_id` / `title` / `date` / `text`, or update `fallback_search`
  to match.
- **The documents do not contain the facts that live in §4.** Ideally one
  document points explicitly at the other source. Then an agent that answers
  from the archive looks wrong in the trace.
- Plant at least one **dependency chain**: a document mentions an id, the
  capability resolves it.

Loading from files is fine too, as long as the name `DOCUMENTS` survives.

After you change `DOCUMENTS`, call `rebuild_index()` in the next cell.

In [ ]:
# ==========================================================================
# 🔴 PLACEHOLDER - CORPUS. Replace with this morning's corpus.
# ==========================================================================

DOCUMENTS = [
    {
        "doc_id": "DOC-001",
        "title": "<<REPLACE: an incident record>>",
        "date": "2026-01-01",
        "text": ("<<REPLACE. Describes something that happened and mentions an "
                 "entity by id - without the detail the question will ask for. "
                 "That is what forces a second call to a §4 capability.>>"),
    },
    {
        "doc_id": "DOC-002",
        "title": "<<REPLACE: a procedure or policy>>",
        "date": "2026-01-02",
        "text": "<<REPLACE. A rule or a threshold. Good source for document-only questions.>>",
    },
    {
        "doc_id": "DOC-003",
        "title": "<<REPLACE: a summary or briefing>>",
        "date": "2026-01-03",
        "text": ("<<REPLACE. Ideally says explicitly that the other source is "
                 "authoritative for certain fields, so that picking the wrong "
                 "source is visible in the trace.>>"),
    },
]

print(str(len(DOCUMENTS)) + " documents  (PLACEHOLDER)")

In [ ]:
# PROVIDED - fallback retriever (TF-IDF, no dependencies). Only if this
# morning's retriever is not available.
def _tokenize(text: str) -> list:
    # Latin letters, digits and Hebrew. Extend the class for another language.
    return re.findall(r"[a-z0-9֐-׿]+", text.lower())


def _build_index(documents: list) -> dict:
    doc_tokens = [_tokenize(d["title"] + " " + d["text"]) for d in documents]
    df = Counter()
    for toks in doc_tokens:
        df.update(set(toks))
    n = len(documents)

    def idf(term):
        return math.log((n + 1) / (df.get(term, 0) + 1)) + 1.0

    def vectorize(tokens):
        tf = Counter(tokens)
        vec = {t: (1.0 + math.log(c)) * idf(t) for t, c in tf.items()}
        norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {t: v / norm for t, v in vec.items()}

    return {"df": df,
            "vectorize": vectorize,
            "doc_vecs": [vectorize(toks) for toks in doc_tokens],
            "documents": list(documents)}


_INDEX = None


def rebuild_index(documents: list = None) -> None:
    """(Re)build the index over `documents` (default: the current DOCUMENTS).

    Call this whenever you change DOCUMENTS or change how you chunk them.
    Section 11 invites both, and an index built once when this cell first ran
    would quietly keep answering from the old corpus - a change that appears to
    do nothing is almost always this.
    """
    global _INDEX
    _INDEX = _build_index(DOCUMENTS if documents is None else documents)
    print("index built over " + str(len(_INDEX["documents"])) + " documents")


def fallback_search(query: str, top_k: int = 3) -> list:
    if _INDEX is None:
        rebuild_index()
    df, vectorize = _INDEX["df"], _INDEX["vectorize"]
    qvec = vectorize([t for t in _tokenize(query) if t in df])
    scored = []
    for doc, dvec in zip(_INDEX["documents"], _INDEX["doc_vecs"]):
        score = sum(w * dvec.get(t, 0.0) for t, w in qvec.items())
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda pair: -pair[0])
    return [dict(d, score=round(s, 4)) for s, d in scored[:top_k]]


rebuild_index()
print([h["doc_id"] for h in fallback_search("replace", top_k=3)])

### 3.2 Write the tool: TODO

**The tool returns a string, and that string goes straight into the model's
context.** Whatever you put in it is what it has to think with.

Decisions you are making right now, whether or not you notice:

- How many passages? 3 is a habit, not a law. Try 1. Try 6.
- Include `doc_id`? (Do you want citations? Then yes.)
- The score? (Is the model going to do anything with `0.4127`?)
- Truncate long documents or pass them whole?
- What do you return when there are no results? An empty string is a trap. The
  model will fill the silence. Say it explicitly.

In [ ]:
# TODO - connect this morning's retriever.
@track(name="tool:search_documents")
def search_documents(query: str, top_k: int = 3) -> str:
    """Search the archive. Returns a formatted string.

    TODO:
      1. Call this morning's retriever (or fallback_search) with `query`.
      2. Format the results into a single string. At least doc_id and text.
      3. Handle the zero-results case explicitly.
    """
    raise NotImplementedError("search_documents")


# Check - uncomment once implemented.
# print(search_documents("<<a query that should hit the corpus>>", top_k=2))

> **Exercise 3.a:** print the output for one query at `top_k=1` and at
> `top_k=6`. Estimate the token cost of each. You pay it **on every step of
> every run**. Write down what you chose and why. You come back to this in §11.

---

## 4. Add more sources

The archive is one source. The agent needs at least one more, holding facts that
are not in the documents — otherwise it never has to **choose** a source.

**What that source is, and how you reach it, is up to you.** It might be an
internal client the course already uses, an endpoint, a handle to a store, or a
function somebody else wrote. You are not building it. You are **wrapping it as
a tool**, and that is a different skill.

The distinction matters, because a capability built for programmers is almost
never shaped for a model:

| Built for a developer | What the model needs |
|---|---|
| nested object / raw JSON | flat text, read in one pass |
| throws on a bad id | a sentence saying the id was not found |
| 12 parameters, 3 required | few arguments, all optional |
| returns 400 rows | the few that answer the question |
| a field called `st_cd` | something the description can explain |

The whole right-hand column is the work your wrapper does. None of it is a bug
in the API. It was built for a different reader.

### 4.1 🔴 PLACEHOLDER: the given capabilities

Put here whatever the students receive: imports, client construction,
credentials from the environment, or written functions. However it arrives.

**All this cell has to do is leave two or three callables in scope**, with it
clear what each one returns. Two stubs are provided so the notebook runs.

**Two rules:**

1. **It must hold facts that are in no document**, and ideally a document should
   point at it as the authoritative source for those fields.
2. **There must be a link between the two.** A document mentions an entity by
   id, the capability resolves that id into what the question actually asked
   for. That chain is what makes §9 real.

If the real thing needs credentials or network that may not exist in the
classroom, leave a stub version behind a flag so nobody gets stuck.

In [ ]:
# ==========================================================================
# 🔴 PLACEHOLDER - the capabilities the students are given.
# Replace with a real API client / service call / function.
# ==========================================================================

# 🔴 Whatever setup the real capability needs, for example:
#     from acme_internal import RosterClient
#     roster = RosterClient(base_url=os.environ["ROSTER_URL"],
#                           token=os.environ["ROSTER_TOKEN"])
#
# Credentials from the environment, and read-only if you can: every call here is
# a read, and the arguments are chosen by the agent.

USE_STUB_CAPABILITIES = True   # 🔴 turn off once the real thing is wired in


def lookup_a(**criteria):
    """🔴 PLACEHOLDER - the first capability, exactly as students receive it.

    Rename it to whatever it really is. Document the two things needed to wrap it:
      - what it returns  (a list of records? one record? a response object?)
      - how it fails     (returns empty? throws? returns an error field?)

    The stub returns list[dict] and [] on no match. If the real API throws on a
    miss, or returns a nested envelope, write that here - the wrapper in §4.2
    has to cope with it and students cannot guess.
    """
    if not USE_STUB_CAPABILITIES:
        raise NotImplementedError("wire in the real capability")
    records = [
        {"id": "A-1", "name": "<<REPLACE>>", "status": "<<REPLACE>>"},
        {"id": "A-2", "name": "<<REPLACE>>", "status": "<<REPLACE>>"},
    ]
    return [r for r in records
            if all(str(r.get(k, "")).lower() == str(v).lower()
                   for k, v in criteria.items() if v is not None)]


def lookup_b(**criteria):
    """🔴 PLACEHOLDER - the second capability. Note `a_id`: the link back to the
    first source is what makes multi-step questions possible."""
    if not USE_STUB_CAPABILITIES:
        raise NotImplementedError("wire in the real capability")
    records = [
        {"id": "B-1", "a_id": "A-1", "name": "<<REPLACE>>"},
        {"id": "B-2", "a_id": "A-2", "name": "<<REPLACE>>"},
    ]
    return [r for r in records
            if all(str(r.get(k, "")).lower() == str(v).lower()
                   for k, v in criteria.items() if v is not None)]


print(lookup_a(id="A-1"))
print(lookup_b(a_id="A-1"))

### 4.2 Wrap them as tools: TODO

One tool per capability. The same three decisions as in §3 (name, arguments,
returned string), plus the ones that only show up when you wrap something you
did not write:

- **Which arguments do you expose?** Not all of them. An argument the model
  cannot fill correctly is worse than one that does not exist: it produces
  confident, wrong calls. Few, optional, named the way the question is phrased
  rather than the way the store is.
- **How much of the record do you return?** On one question half the fields are
  noise; on the next, the half you dropped was the answer. A fixed format cannot
  serve both. Decide, and be ready to justify it in §12.
- **What does an empty result look like?** `""` is a trap. Say nothing was
  found, and say what you looked for.
- **What does a failure look like?** If the capability throws or returns an
  error, the wrapper turns it into a sentence the model can act on. §7 catches
  what escapes, but a message written here is far more useful.

In [ ]:
# TODO - wrap each capability as a tool. Rename before students see this.
@track(name="tool:get_a")
def get_a(id: str = None, name: str = None, status: str = None) -> str:
    """Returns a formatted string for the model.

    TODO:
      1. Call lookup_a with whatever criteria arrived.
      2. Turn the result into a single string. Decide which fields earn a place.
      3. Handle "not found" explicitly.
      4. Handle a capability failure - do not let a raw exception escape.
    """
    raise NotImplementedError("get_a")


@track(name="tool:get_b")
def get_b(id: str = None, a_id: str = None, name: str = None) -> str:
    """Returns a formatted string for the model. Same four steps, against lookup_b."""
    raise NotImplementedError("get_b")


# Checks - uncomment once implemented.
# print(get_a(id="A-1"))
# print(get_b(a_id="A-1"))

---

## 5. Tool schemas

The functions exist, but the model has never seen them. What it sees is a
**schema**: a name, a natural-language description, and a JSON Schema for the
arguments.

This is the highest-leverage prose in the system. The model has no other
information about your tools. If two of them sound alike, it will pick the one
whose description sounds more relevant — not necessarily the one you meant.

- Say **when to use it**, not just what it is. *"Search the archive"* is weak.
  *"Search reports, summaries and procedures. Use it to find out what happened
  in an incident or what a procedure requires"* is strong.
- Say **when not to**, when there is another tool it could be confused with.
- Describe **every argument**, with its expected format.
- Keep sharp boundaries between tools. Overlap produces a coin flip.

Schemas are written in a **neutral** shape (`name` / `description` /
`parameters`); `to_provider_tools` from §2.2 converts them.

In [ ]:
# TODO - fill in and improve the schemas. The first is written as an example,
# the other two are stubs.
TOOLS = [
    {
        "name": "search_documents",
        "description": (
            "🔴 PLACEHOLDER - rewrite for your domain, keep the shape. "
            "Search the archive: <<document types>>. Use it to find out "
            "<<which questions>>. The archive does not contain <<the fields "
            "that live in §4>> - those have their own tools."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string",
                          "description": "A natural-language query, e.g. '<<REPLACE>>'."},
                "top_k": {"type": "integer",
                          "description": "How many documents to return. Default 3."},
            },
            "required": ["query"],
        },
    },
    {
        # TODO: a good name. A description that says when to use it and when not.
        #       A description for every argument.
        "name": "get_a",
        "description": "Look up a record.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {"type": "string", "description": "TODO"},
                # TODO: which other filters are worth exposing?
                # An argument the model cannot fill is worse than no argument.
            },
            "required": [],
        },
    },
    {
        # TODO: all of it.
        "name": "get_b",
        "description": "Look up a record.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "TODO"}},
            "required": [],
        },
    },
]

print([t["name"] for t in TOOLS])

In [ ]:
# PROVIDED - a probe showing which tool the model would pick, without running it.
@track(name="tool_selection_probe")
def probe_tool_choice(question: str, tools=None, system: str = None) -> dict:
    tools = TOOLS if tools is None else tools
    update_trace(name="probe", tags=["probe"], metadata={"question": question})
    resp = call_llm(messages=[make_user_message(question)], tools=tools,
                    system=system or "You are a helpful assistant.")
    return {"question": question,
            "calls": [{"name": c.name, "arguments": c.arguments} for c in resp.tool_calls],
            "text": resp.text[:200]}


# 🔴 PLACEHOLDER - four questions from your domain, one per decision you want to probe.
PROBE_QUESTIONS = [
    "<<REPLACE: should clearly pick a capability>>",
    "<<REPLACE: should clearly pick document search>>",
    "<<REPLACE: ambiguous - the interesting one>>",
    "<<REPLACE: should pick no tool at all>>",
]


def run_probes(tools=None, system=None):
    for q in PROBE_QUESTIONS:
        out = probe_tool_choice(q, tools=tools, system=system)
        picked = ", ".join(c["name"] + str(c["arguments"]) for c in out["calls"]) or "(no tool)"
        print("Q: " + q)
        print("   -> " + picked)

> **Exercise 5.a. Do not skip this one.**
>
> 1. Run `run_probes()` with the raw descriptions. Write down what got picked
>    for each question.
> 2. Open the four traces in Opik. Look at the tool list that was actually sent.
> 3. Improve the two missing descriptions. Specifically: when to use, when not to.
> 4. Run again and compare with step 1.
>
> The question is: **which exact words changed the decision?** Not "it got
> better" — which words. That is the skill.

In [ ]:
run_probes()

---

## 6. One manual cycle

Before writing a loop, run one turn by hand. Five moves, always the same:

1. Call the LLM with the tools
2. Look at what it asked for: name and arguments
3. Run the Python function
4. Add the result to the conversation
5. Call the LLM again

Nothing is hidden. There is no framework doing step 4 for you, and that is where
most of the bugs live.

In [ ]:
# TODO - one cycle by hand.
# 🔴 PLACEHOLDER: a question that needs a document first, then a capability
#    keyed on what the document returned.
SCRATCH_QUESTION = "<<REPLACE: a two-source question>>"

messages = [make_user_message(SCRATCH_QUESTION)]

# --- step 1: first call -----------------------------------------------------
first = call_llm(messages=messages, tools=TOOLS, system="You are a helpful assistant.")
show(first)

# --- step 2: inspect the request --------------------------------------------
# TODO: pull the first ToolCall out of first.tool_calls. What are .name, .arguments, .id?
call = ...              # TODO
print("name:", ...)     # TODO
print("arguments:", ...)  # TODO
print("id:", ...)       # TODO

In [ ]:
# --- step 3: run the function -----------------------------------------------
# TODO: manual dispatch this time - if the name is "search_documents", call
#       search_documents(**call.arguments), and so on.
observation = ...       # TODO  (a string)
print(observation[:800])

In [ ]:
# --- step 4: append both the assistant turn and the result ------------------
# The assistant turn goes back as-is. Do not rebuild it from first.text - that
# drops the tool-call ids and thinking blocks the provider returned.
messages.append(first.assistant_message)

# TODO: build a ToolResult and append the messages it produces.
#       result = ToolResult(tool_call_id=call.id, name=call.name,
#                           content=observation, is_error=False)
#       messages.extend(make_tool_result_messages([result]))
...                     # TODO

# --- step 5: second call ----------------------------------------------------
second = call_llm(messages=messages, tools=TOOLS, system="You are a helpful assistant.")
show(second)

> **Exercise 6.a:** open the route in Opik: two LLM calls, one tool span between
> them.
>
> - Did the second call answer, or ask for **another tool**? If it asked, that is
>   exactly the multi-step dependency, and exactly why you need a loop rather
>   than two hard-coded calls.
> - What is the physical difference between the input of call 1 and call 2?
>   (Answer: more messages. The growing message list **is** the agent's memory.
>   There is nothing else.)

---

## 7. Tool dispatch

The model returns a tool name **as a string**. You have to turn that into a
Python call, and survive the model getting it wrong — because it will.

Three failure modes. Each one must return **a string the model can read and
recover from**, not raise:

| Failure | Example | Why it happens |
|---|---|---|
| unknown tool | `"search_docs"` | the model half-remembered the name |
| invalid arguments | `get_b(unit="A-1")` | wrong keyword |
| execution error | the API is down, the call throws | bad input or a bug in the tool |

An escaping exception kills the run. A clear error string lets the agent try
again on the next step, and that is real behaviour you will see in the traces.

In [ ]:
# PROVIDED - the name -> function lookup, resolved lazily on purpose.
#
# Section 11 asks you to redefine your tool bodies and re-run. If this cell held
# the function objects themselves ({"get_a": get_a, ...}) it would keep pointing
# at the versions that existed when it last ran, and your edits would silently
# have no effect on the evaluation. Looking the name up at call time avoids that.
#
# If you rename a tool in §11, rename it here too.
TOOL_NAMES = ["search_documents", "get_a", "get_b"]


def resolve_tool(name: str):
    """Return the function currently bound to `name`, or None if there is none."""
    if name not in TOOL_NAMES:
        return None
    fn = globals().get(name)
    return fn if callable(fn) else None

In [ ]:
# TODO - implement dispatch.
@track(name="execute_tool")
def execute_tool(tool_name: str, arguments: dict) -> tuple:
    """Run one tool call. Returns (observation_string, is_error).

    TODO:
      1. resolve_tool(tool_name) returns None -> a message naming the bad tool
         and listing TOOL_NAMES, is_error=True.
      2. Invalid arguments -> catch TypeError, return what was wrong and what
         the signature accepts.
      3. Anything else -> catch Exception, return the type and the message. An
         API outage surfaces here too.
      4. Success -> (result, False). Make sure it is a string.
    """
    raise NotImplementedError("execute_tool")

In [ ]:
# PROVIDED - tests for execute_tool. All four must behave sensibly.
def _test_execute_tool():
    cases = [
        ("happy path",   "get_a",        {"id": "A-1"}),
        ("unknown tool", "search_docs",  {"query": "anything"}),
        ("bad argument", "get_b",        {"nonexistent_kwarg": "x"}),
        ("empty result", "get_a",        {"id": "NO-SUCH-ID"}),
    ]
    for label, name, args in cases:
        out, is_err = execute_tool(name, args)
        print("--- " + label + "  (is_error=" + str(is_err) + ")")
        print("    " + str(out)[:220].replace("\n", " | "))
        assert isinstance(out, str), label + ": observation must be a string"
    print("\nAll four returned a string without raising.")


# _test_execute_tool()

---

## 8. The agent loop

The main event. Everything so far was parts; this is the machine.

Stripped of API detail, an agent is three lines:

```text
while not done:
    action      = LLM(state)
    observation = environment(action)
    state      += observation
```

- **state** is the message list. That is all "memory" means here.
- **action** is a tool call (or text, when it is finished).
- **environment** is your `execute_tool`.
- **done** is "the model returned no tool calls". The model decides when it has
  enough, and that is precisely what makes this an agent rather than a pipeline.

Keep the implementation to one screen. Past that you are building a framework,
which is the thing we are avoiding today.

In [ ]:
# 🔴 PLACEHOLDER - a basic system prompt. Rewrite for your domain.
# Students tune this in §11, so keep it deliberately plain - too strong a prompt
# leaves them nothing to improve.
SYSTEM_PROMPT = """You are an assistant for <<REPLACE: the role and organisation>>.

You have tools for searching the archive and for <<REPLACE: the other sources>>.
Use them when the answer depends on those records. Answer directly, without a
tool, when the question is general knowledge or a writing task.

Ground every factual claim in tool output. If the information is not in any
source, say so explicitly instead of guessing."""

In [ ]:
# TODO - implement the loop.
@track(name="run_agent")
def run_agent(question: str, max_steps: int = 5, tools=None, system_prompt: str = None) -> dict:
    """Run the agent until it answers or runs out of steps.

    Returns: {"question", "answer", "steps", "tool_calls", "stop_reason", "messages"}
    where each item in tool_calls is
      {"step", "name", "arguments", "observation", "is_error"}
    The evaluation in §10 depends on those keys. Keep them.
    """
    tools = TOOLS if tools is None else tools
    system_prompt = SYSTEM_PROMPT if system_prompt is None else system_prompt

    messages = [make_user_message(question)]
    trace = {"question": question, "answer": None, "steps": 0,
             "tool_calls": [], "stop_reason": None, "messages": messages}

    for step in range(max_steps):
        trace["steps"] = step + 1
        response = call_llm(messages=messages, tools=tools, system=system_prompt)

        # TODO: append the assistant turn to messages.
        #       response.assistant_message as-is. Do not rebuild it from text.

        # TODO: if response.tool_calls is empty, this is the final answer.
        #       trace["answer"] = response.text, trace["stop_reason"] = "answered",
        #       and return trace.

        # TODO: otherwise run every call in order. For each: execute_tool, record
        #       a dict in trace["tool_calls"] with
        #       step / name / arguments / observation / is_error,
        #       and build a ToolResult carrying that call's .id.

        # TODO: append the results with
        #       messages.extend(make_tool_result_messages(results))
        #       Pass all of the turn's results in one call - the adapter decides
        #       whether the provider wants one message or several.

    # TODO: handle running out of steps.
    #       trace["stop_reason"] = "max_steps".
    return trace

---

## 9. Single-tool, then multi-step

Start with questions that need one call. Once those work, move to the ones where
**the argument for the second call exists only in the output of the first**:

```text
search_documents("<<the event>>")
  -> document mentions A-1
get_a("A-1")
  -> record mentions B-1
get_b("B-1")
  -> answer
```

You cannot write that chain in advance. The model discovers it. The loop just
keeps feeding observations back until it stops asking.

After every run: **open the trace.** Ask whether the route made sense, not only
whether the answer was right. A correct answer from a bad route is a bug that
has not surfaced yet.

In [ ]:
# 🔴 PLACEHOLDER - warm-up. One tool, one step.
for q in [
    "<<REPLACE: answered from the documents alone>>",
    "<<REPLACE: answered from one capability call>>",
]:
    print_trajectory(run_agent(q))
    print("=" * 78)

In [ ]:
# 🔴 PLACEHOLDER - multi-step. The second call depends on the first observation.
for q in [
    "<<REPLACE: a document, then a capability keyed on the result>>",
    "<<REPLACE: another chain, ideally three steps deep>>",
]:
    print_trajectory(run_agent(q))
    print("=" * 78)

> **Exercise 9.a:** for the deepest chain, open the trace and write down the
> exact sequence of calls. How many steps? Could it have been fewer? Was there a
> redundant call (same tool, same arguments, twice)?
>
> **Exercise 9.b:** run a question that should need no tool at all. Did it call
> something? If so, that is a wasted call and a real defect. Note it; you fix it
> in §11.

---

## 10. Evaluation

One question at a time tells you nothing. You need a dev set and more than one
number.

**Why accuracy alone is not enough.** An agent that calls every tool on every
question will score well on accuracy and be unusable: slow, expensive,
impossible to debug. An agent that answers correctly via the **wrong** route
will break the moment the question shifts slightly. So you measure:

| Metric | What it catches |
|---|---|
| answer accuracy | is the answer right |
| required-tool usage | did it consult the sources it had to |
| unnecessary tool calls | is it calling things it does not need |
| avg tool calls | cost and latency |
| tool execution errors | broken schemas, bad arguments, brittle tools |

Every evaluation run stamps the example id onto the Opik trace. When a number
moves, you go straight to the route that moved it.

### 10.1 🔴 PLACEHOLDER: dev set

10–15 examples, derived from §1, §3 and §4. One row is written out as a template.

**Required coverage**: at least one example per category:

| Category | Purpose |
|---|---|
| `rag_only` | documents alone |
| `structured_only` | one capability call |
| `rag_plus_structured` | a document, then a capability keyed on the result |
| `multi_structured` | two capability calls |
| `multi_step` | three or more dependent calls |
| `no_tool` | general knowledge or writing. Any tool call is a defect |
| `unavailable` | the answer exists nowhere; a healthy agent says so and does not invent |

**Fields**

| Field | Meaning |
|---|---|
| `required_tools` | must be called at least once |
| `allowed_tools` | any call outside this list counts as unnecessary |
| `must_include` / `must_not_include` | substring checks (case-insensitive) |
| `judge` | a criterion for the LLM judge, where substrings do not work |

Use `must_include` when a correct answer contains a specific token: a name, an
id, a number. Use `judge` for `no_tool` and `unavailable`, where correctness is
a **shape** ("declined to invent a number") rather than a particular word.

**Deliberate near-misses are the most valuable rows you will write.** An example
where the obvious query returns only half of what is needed, or where two tools
both sound plausible, is what turns §12 from a ritual into real diagnosis. Write
two or three on purpose, and mark them in the instructor copy so they are not
mistaken for broken rows.

In [ ]:
# ==========================================================================
# 🔴 PLACEHOLDER - DEV SET. Replace every row.
# ==========================================================================

DEV_SET = [
    # ---- full template ----------------------------------------------------
    {
        "id": "ev01",
        "category": "rag_plus_structured",
        "question": "<<REPLACE: a document, then a capability>>",
        "required_tools": ["search_documents", "get_a"],
        "allowed_tools": ["search_documents", "get_a", "get_b"],
        "must_include": ["<<REPLACE: a token the answer must contain>>"],
        "must_not_include": [],
    },
    # ---- stubs: one per remaining category ---------------------------------
    {"id": "ev02", "category": "rag_only",
     "question": "<<REPLACE>>",
     "required_tools": ["search_documents"], "allowed_tools": ["search_documents"],
     "must_include": ["<<REPLACE>>"]},

    {"id": "ev03", "category": "structured_only",
     "question": "<<REPLACE>>",
     "required_tools": ["get_a"], "allowed_tools": ["get_a"],
     "must_include": ["<<REPLACE>>"]},

    {"id": "ev04", "category": "multi_structured",
     "question": "<<REPLACE>>",
     "required_tools": ["get_a", "get_b"], "allowed_tools": ["get_a", "get_b"],
     "must_include": ["<<REPLACE>>"]},

    {"id": "ev05", "category": "multi_step",
     "question": "<<REPLACE: three dependent calls>>",
     "required_tools": ["search_documents", "get_a", "get_b"],
     "allowed_tools": ["search_documents", "get_a", "get_b"],
     "must_include": ["<<REPLACE>>"]},

    {"id": "ev06", "category": "no_tool",
     "question": "<<REPLACE: general knowledge or writing>>",
     "required_tools": [], "allowed_tools": [],
     "judge": "<<REPLACE: what a correct answer looks like. No call is required.>>"},

    {"id": "ev07", "category": "unavailable",
     "question": "<<REPLACE: something documented in no source>>",
     "required_tools": [], "allowed_tools": ["search_documents", "get_a", "get_b"],
     "judge": "The answer states the information is not available in the sources. "
              "It must not invent a value."},

    # 🔴 Add 3-8 more. Target: 10-15 in total.
]

print(str(len(DEV_SET)) + " dev examples  (PLACEHOLDER)")
for cat in sorted({e["category"] for e in DEV_SET}):
    print("  " + cat + ": " + str(sum(1 for e in DEV_SET if e["category"] == cat)))

In [ ]:
# PROVIDED - scoring. Domain-independent.
_VERDICT_RE = re.compile(r"\b(PASS|FAIL)\b")


def _parse_verdict(text: str) -> bool:
    """Read the grader's verdict from anywhere in its reply.

    Deliberately not text.startswith("PASS"). §11 lets you change LLM_SETTINGS,
    and a model switched into a thinking or reasoning mode puts its reasoning
    first - a prefix check would then score every answer FAIL, a collapse that
    looks like an agent regression and is not one. The last verdict token wins,
    so a verdict stated after reasoning is read correctly.
    """
    hits = _VERDICT_RE.findall((text or "").upper())
    if not hits:
        print("!! judge returned no PASS/FAIL verdict: " + repr((text or "")[:150]))
        return False
    return hits[-1] == "PASS"


@track(name="llm_judge")
def llm_judge(question: str, answer: str, criterion: str) -> bool:
    prompt = (
        "You are grading one answer from an assistant.\n\n"
        "QUESTION:\n" + question + "\n\n"
        "ANSWER:\n" + str(answer) + "\n\n"
        "CRITERION:\n" + criterion + "\n\n"
        "Reply with exactly one word: PASS or FAIL."
    )
    resp = call_llm(messages=[make_user_message(prompt)],
                    system="You are a strict but fair grader.")
    return _parse_verdict(resp.text)


def grade_answer(example: dict, answer) -> tuple:
    """Returns (is_correct, reason)."""
    if not answer:
        return False, "empty answer"
    low = str(answer).lower()
    for needle in example.get("must_include", []):
        if needle.lower() not in low:
            return False, "missing '" + needle + "'"
    for needle in example.get("must_not_include", []):
        if needle.lower() in low:
            return False, "contains forbidden '" + needle + "'"
    if example.get("judge"):
        if not llm_judge(example["question"], answer, example["judge"]):
            return False, "judge: FAIL"
    return True, "ok"

In [ ]:
# PROVIDED - runner. Every example stamps its id onto the Opik trace.
@track(name="eval_example")
def evaluate_one(example: dict, **agent_kwargs) -> dict:
    update_trace(
        name="eval::" + example["id"],
        tags=["eval", example["id"], example["category"]],
        metadata={"example_id": example["id"], "category": example["category"],
                  "question": example["question"]},
    )
    started = time.time()
    trace = run_agent(example["question"], **agent_kwargs)
    elapsed = time.time() - started

    called = [c["name"] for c in trace["tool_calls"]]
    required = example.get("required_tools", [])
    allowed = set(example.get("allowed_tools", []))

    correct, reason = grade_answer(example, trace["answer"])
    result = {
        "id": example["id"],
        "category": example["category"],
        "correct": correct,
        "reason": reason,
        "required_tools_used": all(t in called for t in required),
        "missing_tools": [t for t in required if t not in called],
        "unnecessary_calls": [t for t in called if t not in allowed],
        "n_tool_calls": len(called),
        "n_tool_errors": sum(1 for c in trace["tool_calls"] if c["is_error"]),
        "steps": trace["steps"],
        "stop_reason": trace["stop_reason"],
        "seconds": round(elapsed, 1),
        "answer": trace["answer"],
        "trace": trace,
    }
    update_trace(metadata={"example_id": example["id"], "correct": correct,
                           "n_tool_calls": result["n_tool_calls"]})
    return result


def evaluate(dev_set=None, **agent_kwargs) -> list:
    dev_set = DEV_SET if dev_set is None else dev_set
    results = []
    for example in dev_set:
        try:
            results.append(evaluate_one(example, **agent_kwargs))
        except Exception as exc:  # noqa: BLE001
            print("!! " + example["id"] + " crashed: " + type(exc).__name__ + ": " + str(exc))
            results.append({"id": example["id"], "category": example["category"],
                            "correct": False, "reason": "crash: " + str(exc),
                            "required_tools_used": False, "missing_tools": [],
                            "unnecessary_calls": [], "n_tool_calls": 0,
                            "n_tool_errors": 1, "steps": 0, "stop_reason": "crash",
                            "seconds": 0.0, "answer": None, "trace": None})
    return results


def report(results: list) -> None:
    n = len(results)
    print("=" * 78)
    print("PER-EXAMPLE")
    print("=" * 78)
    for r in results:
        mark = "PASS" if r["correct"] else "FAIL"
        extra = []
        if r["missing_tools"]:
            extra.append("missing=" + ",".join(r["missing_tools"]))
        if r["unnecessary_calls"]:
            extra.append("extra=" + ",".join(r["unnecessary_calls"]))
        if r["n_tool_errors"]:
            extra.append("tool_errors=" + str(r["n_tool_errors"]))
        print(r["id"] + "  " + mark + "  " + r["category"].ljust(20)
              + " calls=" + str(r["n_tool_calls"]) + " steps=" + str(r["steps"])
              + " " + ("; ".join(extra) or "")
              + ("" if r["correct"] else "   <- " + r["reason"]))

    print("=" * 78)
    print("SUMMARY  (n=" + str(n) + ")")
    print("=" * 78)
    print("answer accuracy          : " + str(round(100 * sum(r["correct"] for r in results) / n, 1)) + "%")
    print("required-tool usage      : " + str(round(100 * sum(r["required_tools_used"] for r in results) / n, 1)) + "%")
    print("examples w/ extra calls  : " + str(sum(1 for r in results if r["unnecessary_calls"])))
    print("avg tool calls / question: " + str(round(sum(r["n_tool_calls"] for r in results) / n, 2)))
    print("total tool errors        : " + str(sum(r["n_tool_errors"] for r in results)))
    print("hit max_steps            : " + str(sum(1 for r in results if r["stop_reason"] == "max_steps")))
    print("avg seconds / question   : " + str(round(sum(r["seconds"] for r in results) / n, 1)))

    print("\nBY CATEGORY")
    for cat in sorted({r["category"] for r in results}):
        rows = [r for r in results if r["category"] == cat]
        acc = round(100 * sum(r["correct"] for r in rows) / len(rows))
        print("  " + cat.ljust(22) + str(acc) + "%  (" + str(len(rows)) + ")")

In [ ]:
baseline_results = evaluate()
report(baseline_results)

---

## 11. Optimization challenge

Improve the agent on the dev set. You may change:

- the system prompt
- tool names and descriptions
- argument schemas
- what the tools return and how it is formatted
- retrieval parameters: `top_k`, chunking
- `max_steps`
- error handling in `execute_tool`
- anything in `LLM_SETTINGS`

Off limits today: explicit planning, a task decomposer, a scratchpad, or a
framework. That is tomorrow.

**Working order:**

1. Run the evaluation
2. Find a failure
3. Open the trace in Opik
4. Find the first wrong decision, not the last sentence
5. Form one hypothesis
6. Change one thing
7. Run again

Two common mistakes:

1. **Fixing the last step.** The answer is wrong because six steps earlier the
   wrong document came back. Rewording the answer will not move anything.
2. **Changing five things at once.** Then the score moves and you have no idea
   what moved it. One thing. Run. Write the number down.

Two things to re-run after you edit, or your change will appear to do nothing:
`rebuild_index()` if you touched `DOCUMENTS` or chunking, and the `TOOL_NAMES`
cell if you renamed a tool.

| # | The failure | First wrong step | Hypothesis | The change | Accuracy before → after |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

In [ ]:
# Workspace. Redefine SYSTEM_PROMPT / TOOLS / the tool bodies above, then re-run.
tuned_results = evaluate()
report(tuned_results)


def compare(before, after):
    b = {r["id"]: r for r in before}
    print("changed examples:")
    for r in after:
        was = b.get(r["id"])
        if was and was["correct"] != r["correct"]:
            arrow = "FAIL -> PASS" if r["correct"] else "PASS -> FAIL"
            print("  " + r["id"] + "  " + arrow + "   " + r["reason"])
    n = len(after)
    print("accuracy: " + str(round(100 * sum(x["correct"] for x in before) / len(before)))
          + "%  ->  " + str(round(100 * sum(x["correct"] for x in after) / n)) + "%")
    print("avg calls: " + str(round(sum(x["n_tool_calls"] for x in before) / len(before), 2))
          + "  ->  " + str(round(sum(x["n_tool_calls"] for x in after) / n, 2)))


compare(baseline_results, tuned_results)

---

## 12. Failure analysis

Pick **at least two failed examples** and work them through properly. The helper
below prints a full route; read it alongside the trace in Opik.

### Failure categories

| Category | What it looks like |
|---|---|
| wrong tool | went to the archive when the answer was in the other source, or the reverse |
| missing tool | needed a second call and did not make it |
| unnecessary tool | called something the question did not require |
| bad arguments | right tool, wrong query / id / filter |
| retrieval failure | right tool and a reasonable query, wrong documents came back |
| tool execution failure | the tool threw, or the call returned an error |
| observation interpretation | the observation was right, the model read it wrong |
| reasoning failure | the observations were right, the conclusion was not |
| premature final answer | stopped while information was still missing |
| hallucination | claimed something that appeared in no observation |

### Template

Fill this in twice.

**Failure 1, id: `____`**

1. **What route did it take?** (calls in order, with arguments)
2. **What was the first wrong step?** (a step number, not "the answer")
3. **Which failure category?** (from the table above)
4. **Why did it happen?** (what in the system caused it: a description, a schema,
   a format, the prompt, retrieval parameters)
5. **What change might fix it?** (specific. "Improve the prompt" is not an answer.)

---

**Failure 2, id: `____`**

1. **What route did it take?**
2. **What was the first wrong step?**
3. **Which failure category?**
4. **Why did it happen?**
5. **What change might fix it?**

In [ ]:
# PROVIDED - print the full route for one example.
def inspect_result(results: list, example_id: str) -> None:
    row = next((r for r in results if r["id"] == example_id), None)
    if row is None:
        print("no such example: " + example_id)
        return
    print("=" * 78)
    print(row["id"] + "  [" + row["category"] + "]  "
          + ("PASS" if row["correct"] else "FAIL: " + row["reason"]))
    print("=" * 78)
    if row["trace"] is None:
        print("(crashed - no trajectory)")
        return
    trace = row["trace"]
    print("Q: " + trace["question"] + "\n")
    for call in trace["tool_calls"]:
        flag = "  [ERROR]" if call["is_error"] else ""
        print("STEP " + str(call["step"]) + "  " + call["name"] + flag)
        print("  args: " + json.dumps(call["arguments"], ensure_ascii=False))
        print("  obs : " + str(call["observation"])[:900])
        print("")
    print("stop_reason: " + str(trace["stop_reason"]) + "   steps: " + str(trace["steps"]))
    print("\nANSWER:\n" + str(trace["answer"]))


def failed(results: list) -> list:
    return [r for r in results if not r["correct"]]


# This section needs at least two failures. If your tuning in §11 worked,
# tuned_results may not have two left - so top up from the baseline run.
to_analyse = [("tuned", r) for r in failed(tuned_results)]
if len(to_analyse) < 2:
    seen = {r["id"] for _, r in to_analyse}
    topup = [("baseline", r) for r in failed(baseline_results) if r["id"] not in seen]
    if topup:
        print("only " + str(len(to_analyse)) + " failure(s) left after tuning; "
              + "adding " + str(len(topup)) + " from the baseline run\n")
    to_analyse += topup

if not to_analyse:
    print("No failures in either run. Either the dev set is too easy - add the "
          "deliberate near-misses from §10.1 - or there is a route problem the "
          "grader cannot see. Pick two passing examples and audit their routes "
          "instead.")
else:
    for source, row in to_analyse:
        print("[" + source + " run]")
        inspect_result([row], row["id"])

---

## 13. Final reflection

Short answers. First, get out the predictions you wrote in *Before you start*.

**0. Which of your predictions (13–15) were wrong?**
For each one you got wrong, say what you had assumed that turned out not to
hold. This is the question with the most in it.

**1. What makes this an agent rather than ordinary RAG?**
Both retrieve and then generate. Point at the line in your own code where the
difference lives. Compare with what you wrote for questions 1 and 2.

**2. What determines the behaviour, apart from the model?**
You never changed model, and the behaviour changed a lot. List what you did
change, and rank the top three by how far they moved the metrics. Compare with
your ranking in question 15.

**3. Why is accuracy alone not enough?**
Describe an agent that would score 100% accuracy and still not deserve to ship.

**4. How did traces help you debug?**
One concrete failure you would not have diagnosed from the final answer alone.

**5. What gets hard when far more calls are needed?**
Your loop copes with 3–4. Imagine a task needing 40, over half an hour, where a
mistake at call 6 only surfaces at call 30.

- Where does the message list stop being usable memory?
- How would you know **which** of the 40 steps was the wrong one?
- The model decides its next step from the whole history. What happens to that
  decision when the history is 100k tokens of tool output?
- What would you want the agent to do **before** step 1 that it does not do today?

That last point is tomorrow's exercise: **explicit planning**, memory
management, and long-horizon agents.